# 🚀 Dewatermark Pro — Google Colab GPU Backend Server (v2.2 Precision AI Edition)

Akselerasi GPU Cloud untuk **SimpleLaMa Neural Inpainting** dan Pembersihan Watermark Presisi Tinggi.

### ⚡ UPDATE TERBARU (Sakerly, RFans Precision & Unified Batch Download):
- **Batch Download & Unified ZIP**: Seluruh hasil proses batch otomatis disimpan di folder `/content/cleaned_photos/` di Google Colab dan disatukan ke dalam **1 file ZIP tunggal** (`/content/cleaned_batch_latest.zip` & `/content/{batch_id}.zip`). Tidak ada lagi file ZIP yang terpisah-pisah!
- **Anti-Gagal Download**: Web Studio kini dilengkapi tombol unduh per-foto (`⬇️ Unduh`) langsung dari memori browser, serta tombol `Download Semua (ZIP)` menggunakan client-side JSZip instan dan server fallback.
- **Stempel Bulat RFans**: 100% menggunakan master template bersih `wm_bulat_besar.png` (stroke-level 1-2px, anti-blobs, tekstur karpet & sepatu tetap 100% utuh tanpa kabut buram).
- **Sakerly Signature**: Full-image pyramid search untuk tanda tangan coral-red `t.me/Sakerly` di mana saja (kacamata hitam, rambut, baju, sudut) dalam <20ms.
- **Anti-Solid Box**: Poligon balok solid OCR telah dihapus total. Area stempel bulat dilindungi dari poligon OCR.
- **Zero Training Required**: Template matematis langsung aktif otomatis tanpa perlu training lagi.

### 🚀 Cara Menjalankan (Hanya 2 Langkah Cepat!):
1. **Langkah 1**: Klik menu **Runtime** ➔ **Change runtime type** ➔ Pilih **T4 GPU** ➔ **Save**.
2. **Langkah 2**: Jalankan **CELL 1 (Setup Modul)** (~15 detik).
3. **Langkah 3**: Jalankan **CELL 2 (Start GPU Server)**.
4. Copy link `https://...trycloudflare.com` yang muncul di bagian bawah, lalu tempel ke Web Studio lokal Anda di menu **⚡ GPU Acceleration**!


In [ ]:
# CELL 1: Siapkan Modul LaMa Inpainting GPU (~15 detik)
!pip install --no-deps simple-lama-inpainting ultralytics --quiet
!pip install flask-cors easyocr --quiet

import os, sys
print("✅ Modul LaMa GPU Acceleration berhasil terpasang di Colab!")


In [ ]:
"""
colab_server.py - Google Colab GPU Backend Server (Precision AI Clean Edition)
Anti-False-Positive Engine: YOLOv11 Watermark Detector + Multi-Pass CLAHE EasyOCR + Precision Stroke Masking
Dedicated RFans Master Stamp (wm_bulat_besar.png) & Sakerly Signature (<15ms)
Menghapus Bersih Watermark di Lokasi Acak Manapun TANPA Merusak Kipas, Bunga, Sepatu, atau Karpet.
"""
import os, sys, time, re, subprocess, base64, zipfile, io

# 0. Auto-Installer Instan di GPU Colab (~5 detik)
for pkg, mod in [
    ("simple-lama-inpainting", "simple_lama_inpainting"),
    ("ultralytics", "ultralytics"),
    ("flask-cors", "flask_cors"),
    ("easyocr", "easyocr")
]:
    try:
        __import__(mod)
    except ImportError:
        print(f"[*] Menyiapkan modul AI {pkg} di GPU...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--prefer-binary", "--quiet"])

# Siapkan cloudflared resmi jika belum ada
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

# Siapkan bobot AI YOLO Watermark jika belum ada
if not os.path.exists("yolo11_watermark.pt"):
    print("[*] Mengunduh model AI YOLO Watermark (~109 MB, 1-2 detik)...")
    subprocess.run(["wget", "-q", "-nc", "https://huggingface.co/spaces/fancyfeast/joycaption-watermark-detection/resolve/main/yolo11x-train28-best.pt", "-O", "yolo11_watermark.pt"])

import torch, cv2, numpy as np
from PIL import Image
from flask import Flask, request, jsonify, send_file, Response
from flask_cors import CORS
from simple_lama_inpainting import SimpleLama

# 1. Cek Ketersediaan GPU CUDA
print("=" * 65)
if torch.cuda.is_available():
    print(f"🚀 CUDA GPU Aktif: {torch.cuda.get_device_name(0)}")
    print(f"⚡ VRAM Tersedia: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    USE_GPU = True
else:
    print("⚠️ PERINGATAN: GPU tidak terdeteksi! Pastikan Runtime diatur ke T4 GPU.")
    USE_GPU = False
print("=" * 65)

# 2. Inisialisasi Model Inpainting SimpleLaMa
print("[*] Memuat SimpleLaMa Inpainting ke GPU...")
lama = SimpleLama()
print("✅ SimpleLaMa Inpainting siap di GPU!")

# 3. Inisialisasi Model AI Deteksi Watermark
yolo_wm = None
custom_model_paths = [
    "/content/drive/MyDrive/DewatermarkAI/custom_watermark.pt",
    "custom_watermark.pt",
    "/content/custom_watermark.pt",
    "yolo11_watermark.pt",
    "/content/yolo11_watermark.pt"
]

for p in custom_model_paths:
    if os.path.exists(p):
        try:
            from ultralytics import YOLO
            yolo_wm = YOLO(p)
            if "custom_watermark.pt" in p:
                print(f"🎯 MODEL HASIL TRAINING AKTIF ({p})! (Deteksi Khusus Album Anda)")
            else:
                print(f"✅ Model Pretrained Aktif ({p} run)!")
            break
        except Exception as e:
            print(f"ℹ️ Model load notice for {p}: {e}")

# 4. Inisialisasi EasyOCR GPU untuk Deteksi Teks Watermark Acak (CRAFT)
reader = None
try:
    import easyocr
    reader = easyocr.Reader(['ch_sim', 'en'], gpu=USE_GPU)
    print("✅ EasyOCR Deep Learning Text Detector aktif di GPU!")
except Exception as e:
    print(f"ℹ️ EasyOCR notice: {e}")

app = Flask(__name__)
CORS(app)

def bgr_to_base64(img_bgr, ext='jpg', quality=95):
    if ext == 'jpg':
        _, buf = cv2.imencode('.jpg', img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        mime = 'image/jpeg'
    else:
        _, buf = cv2.imencode('.png', img_bgr)
        mime = 'image/png'
    return f'data:{mime};base64,' + base64.b64encode(buf).decode('utf-8')

def base64_to_bgr(b64_str):
    if ',' in b64_str:
        b64_str = b64_str.split(',', 1)[1]
    return cv2.imdecode(np.frombuffer(base64.b64decode(b64_str), np.uint8), cv2.IMREAD_COLOR)

# -------------------------------------------------------------------------
# MASTER TEMPLATES (Embedded High-Res Base64)
# -------------------------------------------------------------------------
B64_STAMP_TEMPLATE = "..."
B64_SAKERLY_TEMPLATE = "iVBORw0KGgoAAAANSUhEUgAAAgsAAADeCAYAAACzM0N7AAAgAElEQVR4AezBvW4ja5Yg2hUYqzHftoc0B/MU8QDhETipcfu+3W17lGlHObT4KBTa3d/gmnELFYXWUUnKZFA/yTy51xqOh9EbNQxWC7pSSiml/GUMx8PoDXb4imbVcYeOBV0ppZRSfmnD8TC6UmDG3lMdCzru0D1a0JVSSinllzEcD6Mr7TEjvK5j8ahjQiqllFLKL2E4HkZXCMzY2yYxIpVSSinllzAcD6MrBE4I2yRGpFJKKaX8EobjYXSFPWaEbRIjUimllFJ+CcPxMPqOZtWtGgJfsbPdGRNSKaWUUn4Jw/EwekVgtrqzukeg2S4x4ayUUkopv4zheBi9InBCoFs110uMSKWUUkr5ZQzHw+gVe8wI7+OMCamUUkopv4zheBi9IDBj730kJpyVUkop5ZcyHA+jFwROCO8jMSKVUkop5ZcyHA+jFwROCO8jMSHRlVJKKeWXMRwPo3/REJgR3k9H4g7dakFXSimllJs1HA+jPwnMCDQfo2Ox6rhDx4KulFJKKTdlOB5GfxI4IXyejgUdE1IppZRSbsZwPIz+JHBC+HyJEamUUkopN2M4HkZ/EjghfL7EiFRKKaWUmzEcD6M/CcxoVgOaz3HGhFRKKaWUmzEcD6N/0TBYNcwIHysx4ayUUkopN2U4HkbfETghfKzEiFRKKaWUmzIcD6Pv2GNG+FhnTEillFJKuSnD8TB6RWDG3sdKTDgrpZRSys0ZjofRKwInhI+VGJFKKaWUcnOG42H0isAJYbuOxVMDmucSI9LnaBiwoCullFLKdw3Hw+gVe8wI2yQmdE813KN5qmNC+niBGQ0dd+hWC7pSSimlPDEcD6MXBGbsbZcYkZ5rGDy1oPscgRPCqmOx6rhDt1rQlVJKKb+54XgYvSBwQtjujAnp9uwxI7ysY7Hq+IIHpZRSym9sOB5GLwicELZJTDi7PYEZe5c7Y0IqpZRSflPD8TD6Fw2BGWGbxIh0ewInhMslRqTnGrpVQ1dKKaX8BQ3Hw+hPAjMCzXaJEen2BE4Il0uMSE8F/g/+t9X/wf9GKqWUUv5ihuNh9CeBE8J1zpiQbs8eM8LlEhO6pxrucWd1jzt0Ty3oSimllF/YcDyM/mSPGWG7xISz2xOYsbddx+KpAf8d/9fqv+P/YvFUxx06FnSllFLKL2Y4Hkb/FJixd53EiHR7AieEz9exoOMOia6UUkr5RQzHw+ifAieE7ToSE9JtaQjMCD9XR+ILHpRSSim/gOF4GP1T4ISwTWJCorstgRmB5nY84A88KKWUUm7ccDyM/ilwQtgmMSLdnsAJ4facMSGVUkopN2w4Hkb/tMeMsE1iRLo9gRPC7UmMSKWUUsoNG46H0d8FZuxtlxiRbs8eM8LtSUxIdKuGwY8t6EoppZRPMBwPo7/b4W8I250xId2WwIy929WRuLO6R/NjHXfoWNCVUkopH2Q4HsaGb/hfGGyTmHB2ewInhOt1LJ4b0LyfbtVcrmNBxx261YKulFJKeSfD8TD6ux3+hrBNYkS6PYETwnYdiTt0zzXco2FA83N1LFYdX/CglFJKeQfD8TD6u8AJYZvEhLPbs8eMsE1iQqJ7XcOAhhnhdjzgDzwopZRS3mg4HkZ/FzghbHfGhHQ7AjP2tkuMSJcJnBBuyxkTUimllPIGw/Ew+rvACWG7xIRutaD7uQInhO0SI9JlAieE25KYcFZKKaW8wXA8jP4ucEK4Tsdi1XGHjgXd59tjRtjujAnpMoETwu05Y0IqpZRSrjQcD6O/C8zYex8dCzru0K0WdB8rMGNvu8SEs8sFTgi3JzHhrJRSSrnScDyM/mmPGeF9dSxWHRMSDYPnFnTXC5wQtkuMSJcLnBBu0xkTUimllHKF4XgY/VNgxt7HOWOymtE813GHbrWgu1zghLDdGRPS5faYEbbrWLxsQPN2iQmJ7joNg0cLulJKKb+F4XgY/ckeM8L7S0w4I3BCeFnHYtUxIV0mcELYJjHh7HKBGXvbJSZ0L2uYEd6uIzEhbROY0TzqmJBKKaX85Q3Hw+hPAjMCzftKjEgETgg/lpjQrRZ0r9tjRtgmMSJdLnBC2C4xIr0sMKNhQPM2iRHpxxoGq4YZ4VFiQseCrpRSyl/WcDyM/kVD4Ct23k9iRCJwQrhMx2LVMSE9F5ixt11iRLrcHjPCdmdMSK9rGNAwI1ynIzEhfd8OX9GsBjTPdSzouEP3aEFXSinlL2E4Hkav2OErAs3bnTEhsceMsF1iRHqqITAjbHfGhHSZwIy97RITzi4TOCFsl5iQ6F7XEPiKnW06Fo867tCxoCullPLLGo6H0Xc0BO7RrAY02yQmnBGYsXedxIj0KDAj0GyXmHB2ucAJYbvEiHSZwAlhu8SI9LrAjEDzPjoWdNyhWy3oSiml/DKG42F0gYbBquEezWpA832JEYnACeE6iRHpUeCEcJ3EiHS5wAlhuzMmpMsETgjbJUak1+0xI3yMjsWq4w4dC7pSSik3bTgeRldoGKwa7tE8GtCsOhITEnvMCNt1JCakR3vMCNdJjEiX22NG2CYx4exygRPCdmdMSC8LzNj7PB0LOu7QsaC7PQ2DRwu6Ukr5jQzHw+gdNAweNcxWExIdgRl72yUmJLpHgRl710uMSJcJzNjbLjEiXS5wQtgmMeHsdYETws/RsaBjQrodgRnNo44JqZRSfhPD8TD6AIHZakJaBU4I2yVGpKcCJ4TrnTEhXSZwQtguMSJdLnBC2CYxIr0ucEL4uRITzm5H4ITwKDGhWy3oSinlL2w4HkYfpFl1jwIz9rZLjEhPBU4I10lMOLtc4ISw3RkT0uX2mBG2SYxIL2sIzAg/3wO+INH9XA2BGeGpjsWq4w7dakFXSil/IcPxMPpkO3xF82hA832JEempPWaE7ToSE9Ll9pgRtklMOLtcYMbedokR6bnAjEBzOzoSE9LPscNXBJof61isOr7gQSml/EUMx8PoJ2gYPGq4R6B52RkT0qPAjL3tEhMS3eUCM/a2S4xIlwucELY7Y0J6bo8Z4W06mveVmJDoPtcO37BzvTMmpFJK+QsYjofRjWgI3KNZDWhITDh7KnBC2C4xIm0TOCFsd8aEdLk9ZoRtEhPOngvM2LteR+L/wX+gWQ1o3q4jMSF9jh2+YedtzpiQSinlL2A4HkY3pmGwapitRqSnAieE7RIj0jaBE8I2iQlnlwvM2NsuMSI9FzghXCcxIfH/4d8wWDXMCG+XGJE+XmDG3tskJpyVUspfxHA8jG5YYLaakJ7a4W8I250xIV2uITAjbJMYkS4XOCFslxiRngucEK6TGJGeC8xoGNBcLzEifbw9ZoTrdSQmpG2aVVdKKTdmOB5GN65ZdU81fMP/xH+zTWLC2eUCMwLNdokR6XKBE8J2Z0xIzwVOCNt1JCaklzUMaPiKneskJpx9rMCMves94AsS3esaBk813FvdoWNBV0opN2A4Hka/sP+Bbwg0l0uMSJcLnBCuc8aEdLk9ZoRtEhPOXrbHjLBNYkKiu8weM8J1zpiQPkZDYEa4zgP+wIOXNQxouEfz1IBm1bGg4w7dakFXSik/wXA8jH5xDYF7NI8GNC9LjEiX22NG2C4x4exygRl72yVGpOcCM/a2S4xIlwucEK6TGJHe3w5fEWiuk5hw9rIdvqJhQHO5jsWq4w7dakFXSimfYDgeRn8RDYNHDfdoGNA8OmNCukxgxt51EiPSZRoCM8J2Z0xIz+0xI2yXmNCtFnTfFzghXCcxIr2vHb5h523OmJCe2+Ebdt5Hx2LVcYeOBV0ppXyQ4XgY/YU1DGi4R0PHFzy4XOCEsF1HYkL6scCMQLNdYsLZc4EZe9frWKw67pDoXhY4IVwnMSK9nx2+YedtHvAHHjy3wzfsfJyOBR136FYLulJKeSfD8TD6TTQMWNBtEzghbJOYkOguEzghXCcxIj0XOCG8n47EhPRc4IRwncSI9D4CM/beJjHh7LnAjL3P07FYdUxIpZTyDobjYVR+aI8ZYZvEiHS5wAnhOokR6bnACeF9JSZ0qwXdKnBCuM4ZE9L72GNGuF5HYkJ6LnBC+DkSEzoWdKWU8gbD8TAq3xWYsbddYkS6XOCEcJ0zJqTn9pgR3l/HYtVxh46GGWG7xISz97HDN+xc7wFfkOheFjgh/DwdCzru0K0WdKWUssFwPIzKdwVm7G13xoR0uT1mhO0SE86eC8zY+xwdCwY010mMSG/TEPiKnes94A88+L49ZoTb0LFYddyhY0FXSik/MBwPo/JDO3xFsxrQfF9iwtnlAjP2rpMYkZ4LnBB+DR2JCel6gRmB5noP+AMPvi8wY+92dSzouEPHgq6UUl4wHA+jcpGGwarhHg0DmucSI9LlAieE7ToSE9JTDYEZ4fYlJiS6t9ljRrjeA/7Ag+9rCMwIv4aOBR0TUiml/IvheBiVqzQMaJgRnkqMSJcLnBC2SUxIdE8FZgSaX0NiRHqbwIy96yUmnH1fYEag+fUkRqRSSvkXw/EwKm8SmNGsBvwb/hMT0uUCJ4RtEiPSc4ETwq8jMSK9zR4zwvUSI9L3BU4Iv6bEiFRKKf9iOB5G5c0aBquG/8C/48E2e8wI2yRGpOcCJ4TrdTSf54wJ6Xo7fMPO25wxIX1f4IRwvY7FcwOaj3XGhFRKKf9iOB5G5d01dNsEZuxtd8aE9FzghHCdB/w7/gPNakDzMRITzq63wzfsXK8j8QUPfmyPGeE6D/iC7rmGezQMaN5XYsJZKaW8YDgeRuUmBE4I2yQmnL0scELYLjHhP/FvGKwa7tGsBjTvIzEibdcQ+Iqd6z3gCxLdjwVm7F0nMeHsdQ0DGu7RrAY0b5MYkUop5QXD8TAqNyEwY2+bxIj0ssAJYbvEiPRcw2DVcI9mNaC5zhkT0jY7fEWgud4D/sCDywVOCNdJjEiXaRisGu7RMKDZpiMxIZVSyguG42FUbsYOX9EwoPmxMyaklwVOCNslRqQfaxisGu7RMKC5TGLC2TaBGXtv84A/8GCbwAlhu47EhHSdhgEN92hWA5rXJSYkulJKecVwPIzKTWkY0DAjvC4x4ex1gRPCdokRabuGAQ0zwo8lRqRtAieE6yUmnG0XOCFsk5iQ6N5Hw2DVMCO8LDEilVLKdwzHw6jcpMCMhgHNc4kR6XWBE8J2iQmJ7jqBGc1qQPOyxIi0zR4zwvXOmJC2C5wQtkmMSB8jMKNZDWgeJUakUkr5juF4GJWb1TCg4R7NasC/4T8xIb0ucEK4TkfiDt1qQXe5hsGq4R4NA5pHZ0xIl9vhG3aul5hwdp3ACWGbxIRE9zEaBquGezSrjgmplFK+YzgeRuWX0DBYNfwH/h0Pvi8wI9Bcr2Ox6rhDt1rQXa5hQMM9Gjq+4MHldviGnet1JCak6wROCNt1JCakj9cwWC3oSinlB4bjYVR+SQ3dZRoCM8L76FisOu7QPVrQ/VjDgAXd5Xb4hp3rPeALEt31AieE6yQmJLpSSrkxw/EwKr+FwIy9j9GxeNRxh+7Rgu5tGgJfsXO9B/yBB28XOCFcryNxh44FXSml3IDheBiV38YeM8Ln6Fg86piQrhOYEWiul5hw9j4CJ4S361jQMSGVUspPNhwPo/LbCMzY+zkSEzoWdNvsMSO8zRkT0vsInBDeT2JCt1rQlVLKTzAcD6PyW9nhKwLN5+tY0DEhXWaHb9h5mwf8gQfvJzBj7311LFYdd+hWC7pSSvkEw/EwKr+dhsA9GgY0nysxIv1YYMbe2yQmnL2/PWaEj9OxWHXcoWNB9/GaVVdK+a0Mx8Oo/LYaBjTco2FA8/HOmJB+bI8Z4TodCzompPcXmBFoPkfHgo47JLqPscNXqzt0jxZ0t61h8GhBV0q5yHA8jEr5u4YBDfdoVgOa95WYcPZjO3zDznUSEzoWdB+nIfAVO5+rI3GHjgXd+wjM2Ft1LB513KF7akH38wVmNI867tCtFnSllBcNx8OolH/RMFg13KN5NKC5XmJEel1D4Ct2rnfGhPR5dviGnc/XsaDjDonubQInhNd1LJ7quEO3WtB9vsAJ4amOxarjDh0LulLKfxmOh1EpP9AweNQwI1wnMSK9LDAj0FwvMeHs8+0xI/w8HYkJ6XqBE8J2HYtVx4T0uQInhO/rWNAxIZVS/mE4HkalbBSY0TCg2SYxIr1sjxnhbc6YkD5fYEbDgObnSEw4u17ghPA2iQkdC7rPETghXCYxIdGVUgzHw6iUKzQMaLhH89SA5mVnTEjP7fANO2/zgD/w4OdpGNBwj4YBzed6wBckuu32mBHermNBxx26Rwu697fHjHC5jsSEVMpvbjgeRqW8UcPgqYZ7NE91fMGD53b4hp23ecAfeHA7GgY03KNhQPM5OhIT0uUCM/beX8fiUccdEt37CMzY2y4xIpXymxuOh1EpH6Rh8NSC7rkdvmHneh2JL3hwuxoGNNyjWQ1oPk5iwtnlAieEz9GRuEPHgu56gRPCdmdMSKX85objYVTKTxaYsXe9xIRE9+toGKwa7tEwoHl/Z0xIlwmcED5Xx4KOOyS67faYEbZJTDgrpRiOh1EpP1nghHC9xIj0a2sY0DAjvK/EiHSZwAnh5+lITEiXC8zY2y4xIpVSDMfDqJSfLDAj0FznjAnpryEwo1kNaN4uMSJdJjCjWQ1oPl9iRLpc4ISwXWJEKqUYjodRKTegIXCP5tGA5vse8AcefKxm1X2OhsGq4R7NakCzXWJEulzDYNVwj+bRgOZjJSacXS5wQtjujAmplGI4Hkal3JCGwaOGewSa5x7wBx58rMBsdYdutaD7HA2DVcOMsE1iRLpew+BRwz2aRwOa93XGhHSZPWaEbRITzkop/zAcD6NSblxD4B7No44vePDx9pgR6FisOu7QPVrQfazACWGbxIj0vhoGjxru0TCgebvEhLMfC8zY2y4xIpVS/mE4Hkal/CIaBo8WdB8vMGPvZR2LRx13SHTvryEwI2yTGJE+XsOAhns0qwHNdc6YkL4vcELYLjEilVL+YTgeRqWU7wqcEC7XkZiQ3k9gRqDZLjEifa6GwaphRtguMSJ9X+CEsF1iRCql/MNwPIxKKd8VOCFsk5iQ6N5H4IRwncSI9LqGwfct6K4TOCFslxiRvm+PGWG7MyakUso/DMfDqJTyXYETwnYdiTt0LOiuFzghXCcxIj3XELhH830dd+hY0F0u8DfsbJcYkV4XmLG3XWLCWSnlvwzHw6iU8l17zAjX61jQcYeOBd02e8wI1zljQnpqh68INJfpWNBxh+7Rgu5lA/4XvqHZJjEivS5wQtjujAmplPJfhuNhVEp5VWDG3vvpWNBxh261oHtdYMbedRITzh41BL5i53odi0cdd0h0TwX+hp3tEiPSyxoCM8I2iQlnpZQnhuNhVEp5VeCE8DE6FquOO3Qs6J7aY0a4TmJEWgVmBJr315G4Q/eoYUbY7owJ6bnAjECzXWJEKqU8MRwPo1LKqwIzGgY0H6tjQccdulXDV+xcpyMxIa32mBE+Vsfi0YBmu8SEs5cFTgjXSYxIpZQnhuNhVEr5roYBDTPC5+hYrAY010lMSHSrwIy9X0diRHpZ4IRwncSIVEp5YjgeRqWUiwRm7P1aEiPSo8AJ4deRGJFetseMcJ0zJqRtGgZPLehK+YsYjodRKeViO3xFw4Dm9iVGpEd7zAi/jsSI9Fxgxt51EhPOttnhK5qnOu7QrRZ0pfyihuNhVErZpGFAwz0aBjS36YwJaRWYsfdrOWNCei5wQtiuIzEhXS4wY+9lHYtVxx261YKulF/EcDyMSilXaxjQcI9mNaD5+RITzh4FTgi/jsSEs5ftMSNsk5iQ6C7TMKBhRrhMx2LV8QUPSvkFDMfDqJTyLhoGq4Z7NAxofo4zJqRHe8wI23UsnhrQfKzEiPRcYMbedokR6TI7fEXDgOZ6Z0xIpdy44XgYlVI+RMOAhhnhcyUmnD0KzNjbLjGhe6phRvg4iRHpucAJYbvEiPRjgRl77yMxIpVy44bjYVRK+VCBGQ0Dms9xxoT0KHBC2C4xIj0VmNGsBjTv64wJ6bnACWG7MyakH9tjRni7jsSEVMqNG46HUSnlwzUMaLhHw4DmYyQmnD0VOCFslxiRnmsYrBru0TCgeZvEhLOX7TEjbJOYcPZjO3zDztslJiS61zUMXregK+UTDMfDqJTyqRoGNNyjWQ1o3scZE9JTgRPCdokR6ccaBjR8w/9wvcSI9Fxgxt52iQlnr2sIfMXO23Qs6JiQXtYQuEfzuo47dKsFXSkfYDgeRqWUn6ZhsGq4R8OAZpuOBR1f8OC5wAlhu8SEs8v9N/xPfENznTMmpOcCJ4TrPOALutWCbrXDVwSat3nAF3Qs6F62w1cEmh/rWKw67tCxoCvlnQzHw6iUcjMaBjTco7lMxx06FnQvC5wQrvOAL+hWC7rXBf6GneskJpy9bI8Z4Xodi1XHndVX7LzdA/7Ag+/b4Rt2rtexoOMO3WpBV8qVhuNhVEq5SQ2DyyzofixwQrhex2LVcYfuZQ0zwnUSI9JzgRl776tbNW/3gD/w4HUNga/YeT8di1XHHbrVgq6UCw3Hw6iU8tsInBDeT8fiZQOa6yVGpOcCJ4TblJhw9rrAjEDzsToWq44JqZQLDMfDqJTy2wjM2Lt9HYkJ6bnACeE2nTEhvS5wQvhciQmJrpQfGI6HUSnlt7LHjHC7EhMS3XMNgRnhtnQkvuDB9+0xI3y+jsQdOhZ0pbxgOB5GpZTfSmDG3u1KjEjPBWYEmtvygC9IdN8XmLH3c3Us6PiCB6X8i+F4GJVSfjs7fMPObTpjQnoucEK4HR2JL3hwmT1mhNvxgD/woJQ/GY6HUSnlt7TDVwSa25GYcPaywAnhNjzgCxLdZQIz9m7PA/7Ag1L+aTgeRqWU31ZDYEa4DYkR6WWBE8LPl5hwtk3ghHCbzpiQSvm74XgYlVJ+a4EZDQOanysxIr1sjxnhOh3N+zhjQtomcEK4TsfiqQHN+0hMOPvraBiwoCubDMfDqJTy22sY0HCPZjWg+VxnTEjPBWbsXSdxh/8XDQOa6yQmnG0XOCFs94Av6J5quEfDgOZtzpiQfqxhwILu9gRmNHTcoWNBV35oOB5GpZTyJw2DVcM9mm0GNNslJpy9LHBC2K4jMWHBgIZ7BJptEiPSdnvMCNskJpy9rGFAwz0CzXUSI9L3BWY0dNyhWy3ofr49ZoRVx4KOO3SPFnTlieF4GJVSync0DLZpmBG2SYxIL9v7/9uDe+VI0uwwoCdNhr7rdz7K5zO9CvWArvR4tImhnVZZ+RQKmQnSvcmQJZUYWxODbeKvshoFoHfvOcwI+yQmJDY/agjMCJdLTFjtE5gx2i/Rkd7WEPgd3+yX6EivGzEjnG04Odtwhw0nbD5eYMboeRtOHm24w4YTNsVwPHSllPLOAgvCPismpKcCM0b7JTrS8wILwj4rJqTLBRaE/RId6XIjZoR9Eh3pZYEZo5dtOGHDHRKbjxNYEC634f/h3/Edm79zw/HQlVLKOwvMGF0uMWH1vMCCsN+KCel5gRmjfRId6XIjZoT9Eh3pcoEFYZ9ER3pZYEG4zIbEhPQxRswI+yT+EQ+K4XjoSinlBkbMCJdJdKTnBRaEfRITVq8bMSNcLtGRLhOYMbpOoiNdLrAg7JPoSC8LLAiXS3Sk2wvMGO2X6EjFcDx0pZRyA4EZzaMBzfMSHel5I2aEfRId6XWBBeFyiY50mcCCcJ3EhM3ZCZvXjZgR9kl0pOc1BGaEyyU60tsaNtcLLAj7JTpSMRwPXSml3EjD4FHDPRoGNI9WTEhPBWaM9kt0pNcFFoTLJTrSZQILwvU2nJxtuMOGEzY/CswY7ZfoSE8FZgSafRId6axh8FTDP+N/YnN2wuZygQVhvxUTUjEcD10ppXyghgEN92jY8BsePC+wIOy3YkJ6XWBBuFyiI72tITAjvJ8NJ2y4w+ZRw4yw34oJ6anAgrBfYsKGhns0Tw34B/wfnJxtuMPm7ITNy0bMCPskJqzKXwzHQ1dKKZ+kYcAJm5cFFoR9EhNWbwssCJdLTFi9LjAj0NzOhpNHA5r9EhNWzwssCNfZcMKAZp8NJ2cbJqSnAjNG+yU6UvmL4XjoSinliwssCPskOtLbAgvCPismpJcFFoRfQ6IjPW/EjPB5NiQmpKcCC8J+Kyak8hfD8dCVUsoXF1gQ9kl0pLcFFoR9Eh3pZYEF4deQ6EhPBWaMPk9iQmLzvMCCsE9iwqr8aTgeulJK+eICMxoGNJdZMSG9LTBjtE+iI71sxIzwa0h0pKcCC8LnSXSkl42YEfZJdKTyp+F46Eop5RfQMKDhHs3ZgOapxITV5UbMCJdLdKTnBWaMfh0rJqSnAgvC50l0pOcFZoz2S3Sk8qfheOhKKeUX0zA4a7hHczbgH/DvmJAuF1gQLpfoSM8LLAi/hsSE1fNGzAifJ9GRnhdYEPZLdKTyp+F46Eop5RfXMDhr+Gf8DzzYJ7AgXC4xYfW8ETPCfhtOzgY0t5foSE8FZow+14oJ6amGwIyw34oJqfxpOB66Ukr5G9Ow2S+wIOyzYkL6UWDGaL/EhM1Zwz2aswHN+1sxIT0VWBA+T2LC6qnAjECzX2LCqvxgOB66UkopfxGYMdon0ZF+FFgQ9kt0pEcNg7OGezQMaH5eYsLqeSNmhP02nLxtQPOyFRPSU4EF4TorJqTyg+F46EoppfxpxIxwuURH+lFgQdgv0ZFe1jCgYUb4OYmO9FRgxmi/xITN2xpmhKcSE1bPCywI+yUmrMoTw/HQlVJK+VNgQbhcoiP9KLAg7JfoSG8LzGjOBjT7JTrSU4EFYb9ER3pbYEbDgOZRoiM9b8SMsF+iI5UnhuOhK6WU8qfAjEBzmURH+lFgQdgvMSGxeVvD4KzhHs3ZgOZtKyakpwILwn4rJqTLNAxouEdztmFCeiowY3SdREcqTwzHQ1dKKeUHDYF7NGcDmuclJqx+FL1n1gwAAAllSURBVFgQrrMhcYfE5nINg7OGezQMaJ5KTFg9b8SMsE9iwuo6DYOzEzbPCywI10l0pPLEcDx0pZRSntUwOGu4R3M2oHm0YkJ6FJgx+jkbEnfYPDphc5mGAQ33aH60YUJ6KjBjtF+iI93WiBlhvw2JCak8MRwPXSmllIs0DM4a7tGcbZiQfjRiRvh5G04ebbjDhhM2l2kY/OiEzfMCC8J+Kyak2wnMGO2XmJDYfIyGzS9kOB66UkopV2kYnJ2weSqwINzGhhM2TEjvL7Ag7JOYsLqtwIKwX6IjfYzAv+CfkH4Rw/HQlVJKuZnAjNFtJSZsHp2w+XkjZoR9Eh3ptgILwn6JjnR7DYF73GHDCZsvbjgeulJKKTf1Db8j0NzOhpNHG+6weXTC5nKBGaP9Eh3ptkbMCPutmJBuKzAj8N/wHzhhwx02ZydsvpjheOhKKaXcXEPgd3zzcTacPNpwh8TmbYEFYb8VE9LtBGaM9ktMWN1eYEag+dGGk7MNd0hsvojheOhKKaV8mG/4HYHmc2xITEivCywI+yQmrG4rsCDsl+hIH6MhMCO8bEPiDhtO2Hyi4XjoSimlfKiGwD2aswHNx1kxIb2sITAj7JPoSLc1YkbYb8WE9HECC8LbNpywYUL6JMPx0JVSSvkUDYOzhnsEmttKTFi9LDAj0Oy3YkK6ncCM0X6JCauPFVgQLpfoSJ9kOB66UkopX0JDYEa4nURHellgQdgvMWF1W4EFYb9ER/pYgQXhcomO9EmG46ErpZTyZQRmNI8GNO8n0ZFeFlgQ9kt0pNsKLAj7JTrSxwosCJdLdKRPMhwPXSmllC+lYfCo4R7NowHNdVZMSC8bMSPsl+hIt9MQmBH2WzEhfazAgnC5xITVJxmOh66UUsqX1zB41HCP5mxA87bEhNXLAjNG+21ITEi3EZgRaPZLTFh9vMCCsM+KCel6DQNO2OwwHA9dKaWUX1LD4KzhHs2jAc2PEh3pZYEFYZ/EhMTmdgILwnUSHenjBRaEfVZMSNf5ht/RsOEOG07YvGE4HrpSSil/ExoGjxru0TzaMCG9bMSMsE+iI93WiBnhOismpI8XWBAut+E7/hdO9vuGf8U3jzacsOEOm7MTNv/FcDx0pZRS/mY1DB6dsHlZYMZov0RHup3AjNF1EhNWnyOwIOzzgH9E2icwY/SyDSdnGyakvzIcD10ppZTyh8CCsF+iI91OYEG4TqIjfY7AgrBPoiPtM2JGuExiwoYTNv9pOB66Ukop5Q+BBWG/FRPS7QQWhOskOtLnCCwI+yQ60uUCM0b7bDhhw3f823A8dKWUUsofAgvCPokJq9sKLAj7bUhMSJ8jsCDsk+hIlxsxI1zn/+J/4/twPHSllFLKHwIzmrMBzdtWTEi3NWJG2CcxIbH5PIEFYZ9ER3pbQ+B3fHO9B/x3/NtwPHSllFLKX2kYnDXco2FA81Riwuq2AjNG+yU60ucKzBjtk+hIrwvMCDTXe8B3PPhPw/HQlVJKKa9oGNBwj+ZsQEOiI91WYEHYb8WE9PlGzAiXS3Sk142YEa6XmLD6w3A8dKWUUsqFGgZnDbOzjnRbgQVhn8SE1dcQWBAul+hILwvMGP2cREf6w3A8dKWUUsoVArOzCem2RswI+yQ60tcQWBAul+hILxsxI/ycFRPSH4bjoSullFKu1JxtbiswY7RfYsLm7ITN5wksCJdLdKTnfcO/4puf84DvePBXhuOhK6WUUr64wIJwnQ0nZxvusOGEzccKLAiXS3SkpwIzRtfbkPgND/6L4XjoSimllC9uxIzwPjacsOEOG07Y3F5gQbhcoiM9NWJGuE5iQmLzjOF46EoppZQvLDBjdBsbTthwhw0nbG4jsCBcLtGRHjUEfsc311sxIb1gOB66Ukop5QsLLAi3t+GEDRPS+wssCJdLdKSzwIxAc73EhNUrhuOhK6WUUr6wwILwcRITNmcnbN5HYEG4XKIjnQUWhJ+zYkJ6xXA8dKWUUsoXFlgQPtaGk7MNd9g8OmGzX2BBuFyiI52NmBGu94DvePCG4XjoSimllC8sMKNhQPM5Npw82nCHDSdsLhNYEC6X6Eh8w7/im+s94DseXGA4HrpSSinli2sY0HCP5mxA83k2nLDhDonN6wILwuUS3dmM0fUSE1YXGo6HrpRSSvmFNAzOGu7RMKD5PBsSE9LLAjNGl0tMzmaE62xITEgXGo6HrpRSSvmFNQxouEdzNqD5WImO9LoRM8LlNmfNdR7wGxKbHYbjoSullFL+RjQMzhpmhI+T6EivCywIHyMxYXWF4XjoSimllL9BgRkNA5rbS3Sk1wUWhI+xYkK6wnA8dKWUUsrfqIYBDfdozgY072/FhPS6wIJwew/4jgdXGo6HrpRSSvk70DA4a7hHczag+TmJCau3BRaE20pMWP2E4XjoSimllL9DDYOzhns0DGj2S3SktwUWhNtaMSH9hOF46EoppZSiYUDDPZqzAc3bVkxIbwssCLfzgO948JOG46ErpZRSyg8aBmcNM8LLEhNWlwksCLeRmLB6B8Px0JVSSinlRYEZzdmA5keJjnSZwILw/jYkJqR3MBwPXSmllFJe1TA4a7hH82jDhHSZwILwvh7wGxKbdzIcD10ppZRSdmkYPDphc7nAgvB+EhNW72w4HrpSSimlfKjAgvB+VkxI72w4HrpSSiml/LSGzWUCC8L7eMB3PLiB4XjoSimllPJTAv+Cf0J6W2BB+HmJCasbGY6HrpRSSik/ZcQ97rDhhM3LAgvCz0t0pBsZjoeulFJKKVcLzPiG/8AJG+6w4YTNj0bMCD9vxYR0I8Px0JVSSinlaoEF4UcbTthwh81Zw+/45udsSPyGBzc0HA9dKaWUUq4WWBBetuHkbEDzcx7wGxKbGxuOh66UUkopVxsxI3yMB3zHgw8yHA9dKaWUUq4SmDH6GA/4jgcfaDgeulJKKaVcJbAg3F5iwuqDDcdDV0oppZSrjJgRbi/RkT7YcDx0pZRSStktMGP0MVZMSB9sOB66UkoppewWmDG6nQ0nbPgNDz7BcDx0pZRSSrnKN/yOhgHN+0lM2HDC5pMMx0NXSimllKs1DGi4R8OA5nobEhPSJxuOh66UUkop76JhQMOMsF9iQmLzBQzHQ1dKKaWUdxWY0TCgucyGxIT0RQzHQ1dKKaWUd9cwoOEezes23CGx+UKG46ErpZRSyk01DF53wuYL+v9kNhBiE2b1fQAAAABJRU5ErkJggg=="

_stamp_bytes = base64.b64decode(B64_STAMP_TEMPLATE)
_stamp_raw = cv2.imdecode(np.frombuffer(_stamp_bytes, np.uint8), cv2.IMREAD_UNCHANGED)
if len(_stamp_raw.shape) == 3 and _stamp_raw.shape[2] == 4:
    STAMP_ALPHA = _stamp_raw[:, :, 3]
    _st_mask = (STAMP_ALPHA > 30).astype(np.uint8) * 255
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    STAMP_TOPHAT = cv2.morphologyEx(_st_mask, cv2.MORPH_TOPHAT, k5)
else:
    STAMP_ALPHA = _stamp_raw
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    STAMP_TOPHAT = cv2.morphologyEx(_stamp_raw, cv2.MORPH_TOPHAT, k5)

_sakerly_bytes = base64.b64decode(B64_SAKERLY_TEMPLATE)
SAKERLY_RAW = cv2.imdecode(np.frombuffer(_sakerly_bytes, np.uint8), cv2.IMREAD_UNCHANGED)
SAKERLY_ALPHA = SAKERLY_RAW[:, :, 3] if (len(SAKERLY_RAW.shape) == 3 and SAKERLY_RAW.shape[2] == 4) else SAKERLY_RAW


def detect_sakerly(img_bgr, threshold=0.28, dilation_px=2):
    """
    Dedicated Sakerly Signature Detector.
    Leverages its unique coral-red tint [BGR: 57, 76, 193] and multi-scale template search across
    the entire image (not limited to corners, works on hair, sunglasses, chest, and backgrounds).
    Speed: ~15-30ms with high precision and stroke-level alpha mask.
    """
    h, w = img_bgr.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    th_h, th_w = SAKERLY_ALPHA.shape[:2]
    t_mask = (SAKERLY_ALPHA > 30).astype(np.uint8) * 255

    b, g, r = cv2.split(img_bgr)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h_ch, s_ch, _ = cv2.split(hsv)

    color_gate = (
        (r.astype(int) - b.astype(int) > 16) &
        (r.astype(int) - g.astype(int) > 8) &
        (r > 55) &
        ((h_ch <= 16) | (h_ch >= 165)) &
        (s_ch > 28)
    ).astype(np.uint8) * 255

    scale_search = min(1.0, 1024.0 / float(max(h, w)))
    if scale_search < 1.0:
        gate_search = cv2.resize(color_gate, (0, 0), fx=scale_search, fy=scale_search, interpolation=cv2.INTER_AREA)
    else:
        gate_search = color_gate

    gh, gw = gate_search.shape[:2]
    scales = np.linspace(0.05 * gw / float(th_w), 0.42 * gw / float(th_w), 25)

    best_score = -1.0
    best_match = None

    for s in scales:
        sw = int(round(th_w * s))
        sh = int(round(th_h * s))
        if sw >= gw or sh >= gh or sw < 15 or sh < 8:
            continue
        st = cv2.resize(t_mask, (sw, sh), interpolation=cv2.INTER_AREA)
        res = cv2.matchTemplate(gate_search, st, cv2.TM_CCOEFF_NORMED)
        _, max_v, _, max_l = cv2.minMaxLoc(res)
        if max_v > best_score:
            best_score = float(max_v)
            best_match = (
                int(round(max_l[0] / scale_search)),
                int(round(max_l[1] / scale_search)),
                int(round(sw / scale_search)),
                int(round(sh / scale_search))
            )
            if best_score > 0.82:
                break

    if best_score >= threshold and best_match is not None:
        bx, by, bw, bh = best_match
        st_alpha = cv2.resize(SAKERLY_ALPHA, (bw, bh), interpolation=cv2.INTER_LINEAR)
        _, st_bin = cv2.threshold(st_alpha, 25, 255, cv2.THRESH_BINARY)

        y1, y2 = max(0, by), min(h, by + bh)
        x1, x2 = max(0, bx), min(w, bx + bw)
        roi_gate = color_gate[y1:y2, x1:x2]
        roi_st = st_bin[:(y2 - y1), :(x2 - x1)]

        overlap = np.count_nonzero((roi_gate > 0) & (roi_st > 0))
        template_pts = np.count_nonzero(roi_st > 0)
        ratio = overlap / max(1, template_pts)

        if ratio >= 0.40 or best_score >= 0.50:
            d_px = max(1, dilation_px)
            k_d = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d_px * 2 + 1, d_px * 2 + 1))
            st_bin = cv2.dilate(st_bin, k_d, iterations=1)
            patch = st_bin[:(y2 - y1), :(x2 - x1)]
            mask[y1:y2, x1:x2] = cv2.bitwise_or(mask[y1:y2, x1:x2], patch)
        else:
            best_score = 0.0
            best_match = None

    return mask, best_score, best_match


def detect_stamp_dynamic(img_bgr, threshold=0.34, dilation_px=2):
    """
    Pencarian stempel RFans 100% DINAMIS di 4 Sudut (Top-Left, Top-Right, Bottom-Left, Bottom-Right):
    - Menggunakan master template beresolusi tinggi (wm_bulat_besar.png).
    - Memindai area sudut 35% lebar & tinggi.
    - Menghasilkan HANYA guratan garis stempel halus (1-2px dilation), TANPA merusak karpet / sepatu.
    """
    h, w = img_bgr.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    th_h, th_w = STAMP_TOPHAT.shape[:2]

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    top_img = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, k5)

    cw, ch = int(w * 0.35), int(h * 0.35)
    corners = [
        ('TR', w - cw, 0, w, ch),
        ('BR', w - cw, h - ch, w, h),
        ('TL', 0, 0, cw, ch),
        ('BL', 0, h - ch, cw, h),
    ]

    base_scale = (w * 0.245) / float(th_w)
    scales = np.linspace(base_scale * 0.88, base_scale * 1.12, 7)

    best_score = -1.0
    best_match = None

    for c_name, cx1, cy1, cx2, cy2 in corners:
        c_roi = top_img[cy1:cy2, cx1:cx2]
        for s in scales:
            sw = int(round(th_w * s))
            sh = int(round(th_h * s))
            if sw >= c_roi.shape[1] or sh >= c_roi.shape[0]:
                continue
            st = cv2.resize(STAMP_TOPHAT, (sw, sh))
            res = cv2.matchTemplate(c_roi, st, cv2.TM_CCOEFF_NORMED)
            _, max_v, _, max_l = cv2.minMaxLoc(res)
            if max_v > best_score:
                best_score = float(max_v)
                best_match = (cx1 + max_l[0], cy1 + max_l[1], sw, sh)
                if best_score > 0.65:
                    break
        if best_score > 0.65:
            break

    # Fallback to downscaled scan if not found in standard corners
    if best_score < threshold:
        scale_search = min(1.0, 1024.0 / float(max(h, w)))
        top_search = cv2.resize(top_img, (0, 0), fx=scale_search, fy=scale_search)
        scales_full = np.linspace(0.12 * w / float(th_w), 0.85 * w / float(th_w), 16)
        for s in scales_full:
            sw = int(round(th_w * s * scale_search))
            sh = int(round(th_h * s * scale_search))
            if sw >= top_search.shape[1] or sh >= top_search.shape[0] or sw < 20 or sh < 15:
                continue
            st = cv2.resize(STAMP_TOPHAT, (sw, sh))
            res = cv2.matchTemplate(top_search, st, cv2.TM_CCOEFF_NORMED)
            _, max_v, _, max_l = cv2.minMaxLoc(res)
            if max_v > best_score:
                best_score = float(max_v)
                best_match = (
                    int(round(max_l[0] / scale_search)),
                    int(round(max_l[1] / scale_search)),
                    int(round(sw / scale_search)),
                    int(round(sh / scale_search))
                )

    if best_score >= threshold and best_match is not None:
        bx, by, bw, bh = best_match
        st_alpha = cv2.resize(STAMP_ALPHA, (bw, bh), interpolation=cv2.INTER_LINEAR)
        _, st_bin = cv2.threshold(st_alpha, 25, 255, cv2.THRESH_BINARY)
        d_px = max(1, dilation_px)
        k_d = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d_px * 2 + 1, d_px * 2 + 1))
        st_bin = cv2.dilate(st_bin, k_d, iterations=1)

        y1, y2 = max(0, by), min(h, by + bh)
        x1, x2 = max(0, bx), min(w, bx + bw)
        patch = st_bin[:(y2 - y1), :(x2 - x1)]
        mask[y1:y2, x1:x2] = cv2.bitwise_or(mask[y1:y2, x1:x2], patch)

    return mask, best_score, best_match


def generate_watermark_mask(img_bgr, wm_mode='auto', dilation_px=2):
    """
    Ekstraksi mask watermark presisi tinggi 100% DINAMIS (Zero Hardcoded Coordinates):
    - Beradaptasi otomatis pada setiap gambar yang posisi watermark-nya selalu berubah-ubah.
    - Template Matching dinamis melacak stempel bulat besar RFans di sudut mana saja dengan guratan stroke murni.
    - Sakerly detector melacak signature coral-red di mana saja (<25ms).
    - Deep Learning OCR CRAFT melacak teks transparan miring DENGAN GURATAN STROKE (Zero Solid Blobs).
    - ZERO False Positives: Tidak pernah merusak karpet, bunga, atau sepatu.
    """
    h, w = img_bgr.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    clahe = cv2.createCLAHE(clipLimit=14.0, tileGridSize=(6, 6))

    mode = (wm_mode or 'auto').lower()

    # =========================================================================
    # 1. SPECIALIZED SAKERLY MODE
    # =========================================================================
    if mode in ['sakerly', 'auto']:
        s_mask, s_score, s_match = detect_sakerly(img_bgr, threshold=0.28, dilation_px=dilation_px)
        if s_score >= 0.28:
            mask = cv2.bitwise_or(mask, s_mask)
            if mode == 'sakerly':
                return mask

    # =========================================================================
    # 2. DYNAMIC TEMPLATE MATCHING (Stempel RFans Bulat Besar di 4 Sudut)
    # =========================================================================
    stamp_bbox = None
    if mode in ['rfans', 'auto']:
        stamp_mask, stamp_score, stamp_match = detect_stamp_dynamic(img_bgr, threshold=0.34, dilation_px=dilation_px)
        if stamp_score >= 0.32:
            mask = cv2.bitwise_or(mask, stamp_mask)
            stamp_bbox = stamp_match

    # =========================================================================
    # 3. DEEP LEARNING OCR CRAFT MULTI-ANGLE (Dengan Stroke Masking Murni)
    # =========================================================================
    wm_keywords = [
        'rfans', 'channel', 'bot', 'tg', 'album', 'fans', 'vip', 't.me', '@',
        'sakerly', 'saker',
        '电', '报', '搜', '索', '直', '接', '群', '搜索', '电报', 'telegram'
    ]

    if reader and mode != 'sakerly':
        try:
            lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
            l, a, b_ch = cv2.split(lab)
            l_enh = clahe.apply(l)
            top = cv2.morphologyEx(l, cv2.MORPH_TOPHAT, k5)
            bot = cv2.morphologyEx(l, cv2.MORPH_BLACKHAT, k5)
            amplified_l = cv2.addWeighted(l_enh, 0.75, top, 3.5, 0)
            amplified_l = cv2.subtract(amplified_l, cv2.multiply(bot, 2))
            enh_bgr = cv2.cvtColor(cv2.merge((amplified_l, a, b_ch)), cv2.COLOR_LAB2BGR)

            scale = 1.0
            max_dim = 1280
            if max(h, w) > max_dim:
                scale = max_dim / float(max(h, w))
                small_img = cv2.resize(enh_bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
            else:
                small_img = enh_bgr

            sh, sw = small_img.shape[:2]
            s_center = (sw / 2.0, sh / 2.0)
            scan_angles = [39.0, -39.0, 0.0]

            for angle in scan_angles:
                if angle == 0.0:
                    rot_img = small_img
                    M_inv = None
                else:
                    rad = np.deg2rad(angle)
                    cos_a = np.abs(np.cos(rad))
                    sin_a = np.abs(np.sin(rad))
                    nw = int((sh * sin_a) + (sw * cos_a))
                    nh = int((sh * cos_a) + (sw * sin_a))
                    M_rot = cv2.getRotationMatrix2D(s_center, angle, 1.0)
                    M_rot[0, 2] += (nw / 2.0) - s_center[0]
                    M_rot[1, 2] += (nh / 2.0) - s_center[1]
                    M_inv = cv2.invertAffineTransform(M_rot)
                    rot_img = cv2.warpAffine(
                        small_img,
                        M_rot,
                        (nw, nh),
                        flags=cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_CONSTANT,
                        borderValue=(128, 128, 128)
                    )

                results = reader.readtext(
                    rot_img,
                    contrast_ths=0.02,
                    text_threshold=0.15,
                    low_text=0.10,
                    link_threshold=0.18,
                    mag_ratio=1.2
                )

                for (bbox, text, prob) in results:
                    t_low = text.lower().replace(' ', '')
                    if any(k in t_low for k in wm_keywords):
                        pts = np.array(bbox, dtype=np.float32)

                        if angle != 0.0 and M_inv is not None:
                            pts_orig = []
                            for pt in pts:
                                v = M_inv @ np.array([pt[0], pt[1], 1.0])
                                pts_orig.append(v[:2])
                            pts = np.array(pts_orig, dtype=np.float32)

                        pts = pts / scale
                        bx = int(np.min(pts[:, 0]))
                        by = int(np.min(pts[:, 1]))
                        bw = int(np.max(pts[:, 0]) - bx)
                        bh = int(np.max(pts[:, 1]) - by)

                        # SUPPRESS OCR INSIDE STAMP ROI: detect_stamp_dynamic already masks exact strokes!
                        if stamp_bbox is not None:
                            sx, sy, sw, sh = stamp_bbox
                            ix1 = max(bx, sx)
                            iy1 = max(by, sy)
                            ix2 = min(bx + bw, sx + sw)
                            iy2 = min(by + bh, sy + sh)
                            if ix2 > ix1 and iy2 > iy1:
                                inter_area = (ix2 - ix1) * (iy2 - iy1)
                                box_area = max(1, bw * bh)
                                if (inter_area / float(box_area)) > 0.30:
                                    continue

                        pad = 4
                        x1, y1 = max(0, bx - pad), max(0, by - pad)
                        x2, y2 = min(w, bx + bw + pad), min(h, by + bh + pad)
                        roi = gray[y1:y2, x1:x2]
                        if roi.size > 0:
                            # CRITICAL FIX: Extract ONLY the text strokes (Top-Hat), NEVER solid fill!
                            poly_mask = np.zeros_like(roi)
                            shifted_pts = (pts - np.array([x1, y1])).astype(np.int32)
                            cv2.fillPoly(poly_mask, [shifted_pts], 255)

                            top_roi = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, k5)
                            strk = (top_roi > 12).astype(np.uint8) * 255
                            combined = cv2.bitwise_and(strk, poly_mask)
                            combined = cv2.dilate(combined, k3, iterations=1)
                            mask[y1:y2, x1:x2] = cv2.bitwise_or(mask[y1:y2, x1:x2], combined)
        except Exception as e:
            print(f"[OCR scan notice] {e}")

    # =========================================================================
    # 4. DETEKSI AI YOLOV11 (Bounding Boxes Watermark Acak - Presisi Tinggi)
    # =========================================================================
    if yolo_wm and mode != 'sakerly':
        try:
            results = yolo_wm.predict(img_bgr, conf=0.35, device=0 if USE_GPU else 'cpu', verbose=False)
            total_img_area = float(w * h)

            for box in results[0].boxes:
                bx1, by1, bx2, by2 = map(int, box.xyxy[0].tolist())
                bw, bh = bx2 - bx1, by2 - by1
                if bw <= 0 or bh <= 0: continue

                if (bw * bh) > 0.25 * total_img_area:
                    continue

                if stamp_bbox is not None:
                    sx, sy, sw, sh = stamp_bbox
                    ix1, iy1 = max(bx1, sx), max(by1, sy)
                    ix2, iy2 = min(bx2, sx + sw), min(by2, sy + sh)
                    if ix2 > ix1 and iy2 > iy1:
                        if ((ix2 - ix1) * (iy2 - iy1)) / float(bw * bh) > 0.40:
                            continue

                pad = 4
                px1, py1 = max(0, bx1 - pad), max(0, by1 - pad)
                px2, py2 = min(w, bx2 + pad), min(h, by2 + pad)
                roi = gray[py1:py2, px1:px2]
                if roi.size > 0:
                    top_roi = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, k5)
                    s1 = (top_roi > 12).astype(np.uint8) * 255
                    s1 = cv2.dilate(s1, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
                    mask[py1:py2, px1:px2] = cv2.bitwise_or(mask[py1:py2, px1:px2], s1)
        except Exception as e:
            print(f"[YOLO notice] {e}")

    return mask


@app.route('/api/sample')
def api_sample():
    return jsonify({
        'status': 'ok',
        'device': 'CUDA GPU' if USE_GPU else 'CPU',
        'yolo': yolo_wm is not None,
        'ocr': reader is not None
    })


@app.route('/api/detect', methods=['POST'])
def api_detect():
    data = request.get_json(force=True)
    img = base64_to_bgr(data['image'])
    dilation_px = int(data.get('dilation_px', 2))
    wm_mode = str(data.get('wm_mode', 'auto'))
    mask = generate_watermark_mask(img, wm_mode=wm_mode, dilation_px=dilation_px)
    mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    return jsonify({
        'status': 'ok',
        'mask': bgr_to_base64(mask_bgr, 'png'),
        'pixels_masked': int(np.count_nonzero(mask))
    })


@app.route('/api/inpaint', methods=['POST'])
def api_inpaint():
    data = request.get_json(force=True)
    img_bgr = base64_to_bgr(data['image'])
    h, w = img_bgr.shape[:2]
    wm_mode = str(data.get('wm_mode', 'auto'))

    mask_b64 = data.get('mask')
    if mask_b64:
        if ',' in mask_b64:
            mask_b64 = mask_b64.split(',', 1)[1]
        raw_mask = cv2.imdecode(np.frombuffer(base64.b64decode(mask_b64), np.uint8), cv2.IMREAD_UNCHANGED)
        if len(raw_mask.shape) == 3 and raw_mask.shape[2] == 4:
            mask = raw_mask[:, :, 3]
        elif len(raw_mask.shape) == 3:
            mask = cv2.cvtColor(raw_mask, cv2.COLOR_BGR2GRAY)
        else:
            mask = raw_mask
        if mask.shape != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
    else:
        mask = generate_watermark_mask(img_bgr, wm_mode=wm_mode)

    start_t = time.time()
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    mask_pil = Image.fromarray(mask).convert('L')
    cleaned_pil = lama(img_pil, mask_pil)
    cleaned_bgr = cv2.cvtColor(np.array(cleaned_pil), cv2.COLOR_RGB2BGR)
    elapsed = round(time.time() - start_t, 3)

    return jsonify({
        'status': 'ok',
        'cleaned': bgr_to_base64(cleaned_bgr, 'jpg', quality=98),
        'elapsed_sec': elapsed,
        'dimensions': {'width': w, 'height': h}
    })


# Batch Processing 100+ Gambar (Disimpan Terpusat di Disk Colab & Disatukan dalam 1 ZIP)
BATCH_STORAGE = {}
BATCH_OUTPUT_DIR = "/content/cleaned_photos"
os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

@app.route('/api/batch', methods=['POST'])
def api_batch():
    if 'files' not in request.files:
        return jsonify({'error': 'No files uploaded'}), 400

    files = request.files.getlist('files')
    wm_mode = request.form.get('wm_mode', 'auto')

    # Gunakan batch_id dari client agar semua foto dari antrean tergabung dalam 1 file ZIP!
    batch_id = request.form.get('batch_id')
    if not batch_id or not batch_id.startswith('batch_'):
        batch_id = f"batch_{int(time.time())}"

    # Folder batch khusus di Colab
    batch_dir = os.path.join(BATCH_OUTPUT_DIR, batch_id)
    os.makedirs(batch_dir, exist_ok=True)
    os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

    results = []
    zip_path = f"/content/{batch_id}.zip"
    latest_zip_path = f"/content/cleaned_batch_latest.zip"

    print(f"\n[*] Memproses {len(files)} gambar di GPU [Batch: {batch_id}] (Mode: {wm_mode})...")

    for idx, file in enumerate(files):
        if not file.filename: continue
        file_bytes = np.frombuffer(file.read(), np.uint8)
        img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
        if img_bgr is None: continue
        h, w = img_bgr.shape[:2]

        t0 = time.time()
        mask = generate_watermark_mask(img_bgr, wm_mode=wm_mode)
        img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        mask_pil = Image.fromarray(mask).convert('L')
        cleaned_pil = lama(img_pil, mask_pil)
        cleaned_bgr = cv2.cvtColor(np.array(cleaned_pil), cv2.COLOR_RGB2BGR)
        elapsed = round(time.time() - t0, 3)

        out_name = f"cleaned_{file.filename}"
        out_file_path = os.path.join(batch_dir, out_name)
        root_out_path = os.path.join(BATCH_OUTPUT_DIR, out_name)

        # 1. Simpan foto bersih beresolusi penuh langsung ke disk Colab
        _, buf = cv2.imencode('.jpg', cleaned_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 98])
        img_bytes = buf.tobytes()
        with open(out_file_path, "wb") as f_out:
            f_out.write(img_bytes)
        with open(root_out_path, "wb") as f_out:
            f_out.write(img_bytes)

        # 2. Append ke file ZIP tunggal (agar semua foto batch tergabung dalam 1 file ZIP!)
        for z_target in [zip_path, latest_zip_path]:
            mode_z = 'a' if os.path.exists(z_target) else 'w'
            with zipfile.ZipFile(z_target, mode_z, compression=zipfile.ZIP_DEFLATED) as zip_out:
                existing_names = zip_out.namelist() if mode_z == 'a' else []
                if out_name not in existing_names:
                    zip_out.writestr(out_name, img_bytes)

        # 3. Jika Google Drive tersambung, simpan otomatis juga ke Google Drive
        drive_out = "/content/drive/MyDrive/DewatermarkAI/cleaned_photos"
        if os.path.exists("/content/drive/MyDrive"):
            os.makedirs(drive_out, exist_ok=True)
            with open(os.path.join(drive_out, out_name), "wb") as f_drive:
                f_drive.write(img_bytes)

        # Thumbnail untuk UI antrean
        thumb = cv2.resize(cleaned_bgr, (200, int(200 * h / max(1, w))), interpolation=cv2.INTER_AREA)

        # Base64 foto bersih penuh (seperti mode deteksi single!)
        cleaned_b64 = bgr_to_base64(cleaned_bgr, 'jpg', quality=98)

        results.append({
            'filename': file.filename,
            'out_filename': out_name,
            'elapsed_sec': elapsed,
            'thumb': bgr_to_base64(thumb, 'jpg', 80),
            'cleaned': cleaned_b64,
            'download_url': f"/api/download-file/{batch_id}/{out_name}"
        })
        print(f"  [{idx+1}/{len(files)}] {file.filename} -> Bersih ({elapsed}s) | Disatukan ke: {zip_path}")

    # Update cache memori zip
    try:
        with open(zip_path, "rb") as f_z:
            BATCH_STORAGE[batch_id] = f_z.read()
    except Exception:
        pass

    return jsonify({
        'status': 'ok',
        'processed_count': len(results),
        'batch_id': batch_id,
        'zip_file': f"{batch_id}.zip",
        'items': results
    })


# Endpoint Unduh Foto Individual dari Batch
@app.route('/api/download-file/<batch_id>/<filename>')
def api_download_file(batch_id, filename):
    safe_name = os.path.basename(filename)
    candidates = [
        f"/content/cleaned_photos/{batch_id}/{safe_name}",
        f"/content/cleaned_photos/{safe_name}",
        f"/content/drive/MyDrive/DewatermarkAI/cleaned_photos/{safe_name}"
    ]
    for path in candidates:
        if os.path.exists(path):
            return send_file(path, as_attachment=True, download_name=safe_name)
    return jsonify({'error': 'File not found'}), 404


# Endpoint Unduh ZIP Tunggal Gabungan
@app.route('/api/download-batch/<batch_id>')
def api_download_batch(batch_id):
    zip_path = f"/content/{batch_id}.zip"
    if os.path.exists(zip_path):
        return send_file(zip_path, as_attachment=True, download_name=f"{batch_id}.zip")
    if os.path.exists("/content/cleaned_batch_latest.zip"):
        return send_file("/content/cleaned_batch_latest.zip", as_attachment=True, download_name="cleaned_batch.zip")
    if batch_id in BATCH_STORAGE:
        data_bytes = BATCH_STORAGE[batch_id]
        resp = Response(data_bytes, mimetype='application/zip')
        resp.headers['Content-Disposition'] = f'attachment; filename="{batch_id}.zip"'
        return resp
    return jsonify({'error': 'Batch not found'}), 404


if __name__ == '__main__':
    PORT = 5000
    public_url = "http://127.0.0.1:5000"
    print("\n[*] Menghubungkan ke Cloudflare Tunnel...")
    
    try:
        tunnel_proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        for _ in range(35):
            line = tunnel_proc.stderr.readline()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                public_url = match.group(0)
                break
            time.sleep(0.3)
    except Exception as e:
        print(f"[!] Tunnel notice: {e}")

    print("\n" + "=" * 65)
    print("🎉 GOOGLE COLAB GPU SERVER BERHASIL AKTIF (ZERO FALSE-POSITIVE ENGINE)!")
    print(f"🔗 Public Tunnel URL: {public_url}")
    print("=" * 65)
    print("👉 Copy link 'https://....trycloudflare.com' di atas,")
    print("   lalu buka Web Studio Anda di menu:")
    print("   '⚡ GPU Acceleration' -> centang 'Google Colab GPU' -> Paste URL!")
    print("=" * 65 + "\n")

    app.run(host='0.0.0.0', port=PORT)


---
### 📥 [FITUR BARU] Cek Folder Hasil Foto & Unduh ZIP Langsung dari Colab (Opsional)
Semua foto yang telah dibersihkan oleh server otomatis disimpan di disk Colab `/content/cleaned_photos/` dan disatukan ke dalam satu file ZIP `/content/cleaned_batch_latest.zip`.
Anda dapat menjalankan cell di bawah ini kapan saja untuk melihat file atau mengunduh langsung dari Colab:


In [ ]:
# CELL 3: Cek Folder Foto Bersih & Unduh ZIP Tunggal Langsung dari Colab
import os
from google.colab import files

print("=" * 65)
print("📁 FOLDER PENYIMPANAN COLAB: /content/cleaned_photos")
print("=" * 65)
if os.path.exists("/content/cleaned_photos"):
    items = os.listdir("/content/cleaned_photos")
    print(f"Total file tersimpan: {len(items)} file")
    !ls -lh /content/cleaned_photos
else:
    print("Belum ada foto yang diproses. Silakan jalankan batch dari Web Studio.")

# Unduh ZIP Tunggal Gabungan Hasil Batch Terbaru
if os.path.exists("/content/cleaned_batch_latest.zip"):
    print("\n⬇️ Mengunduh 1 file ZIP gabungan batch terbaru ke komputer Anda...")
    files.download("/content/cleaned_batch_latest.zip")
else:
    print("\n💡 Tip: Jalankan proses Batch di Web Studio terlebih dahulu untuk men-generate file ZIP.")


---
### 📦 [OPSIONAL] Cell Training Tambahan (Tidak Diperlukan untuk RFans / Sakerly)
Hanya jalankan cell di bawah ini jika suatu hari Anda ingin melatih model AI baru untuk watermark asing yang belum ada di daftar template.


In [ ]:
# =========================================================================
# CELL 2: AUTO-TRAIN WATERMARK AI (TRAINING 1 KALI UNTUK SELAMANYA)
# =========================================================================
# Model AI (.pt) akan otomatis disimpan permanen di Google Drive Anda:
# Google Drive/DewatermarkAI/custom_watermark.pt
# Sehingga BESOK, LUSA, ATAU KAPAN PUN ANDA BUKA LAGI, TIDAK PERLU TRAIN LAGI!
# =========================================================================

import os, sys, glob, shutil, random, base64
import cv2, numpy as np

# Ubah ke True jika suatu hari nanti Anda ingin melatih ulang dengan sampel baru
FORCE_RETRAIN = False

DRIVE_DIR = "/content/drive/MyDrive/DewatermarkAI"
DRIVE_MODEL = os.path.join(DRIVE_DIR, "custom_watermark.pt")
LOCAL_MODEL = "custom_watermark.pt"

# Cek apakah model sudah pernah dilatih sebelumnya
already_trained = False
if os.path.exists(DRIVE_MODEL):
    print("=" * 65)
    print("🎉 MODEL CUSTOM DITEMUKAN DI GOOGLE DRIVE!")
    print(f"📁 Path: {DRIVE_MODEL}")
    shutil.copy(DRIVE_MODEL, LOCAL_MODEL)
    print("⚡ Model langsung siap digunakan TANPA PERLU TRAINING ULANG!")
    print("=" * 65)
    already_trained = True
elif os.path.exists(LOCAL_MODEL):
    print("=" * 65)
    print("🎉 MODEL CUSTOM DITEMUKAN DI LOCAL DIRECTORY!")
    print("⚡ Langsung siap digunakan TANPA PERLU TRAINING ULANG!")
    print("=" * 65)
    already_trained = True

if already_trained and not FORCE_RETRAIN:
    print("\n👉 Model sudah aktif! Langsung lewati cell ini dan jalankan CELL 3 (Start Server).\n")
else:
    print("=" * 65)
    print("🚀 MEMULAI AUTO-TRAINING KHUSUS UNTUK WATERMARK ALBUM ANDA...")
    print("⏳ Estimasi waktu: ~2-3 menit di GPU T4 (Gratis)...")
    print("=" * 65)

    # Folder Sampel Watermark Pengguna di Google Drive (Tersimpan Permanen):
    DRIVE_SAMPLES_DIR = os.path.join(DRIVE_DIR, "watermark_samples")
    LOCAL_SAMPLES_DIR = "/content/watermark_samples"
    os.makedirs(DRIVE_SAMPLES_DIR, exist_ok=True)
    os.makedirs(LOCAL_SAMPLES_DIR, exist_ok=True)
    os.makedirs(DRIVE_DIR, exist_ok=True)

    DATASET_DIR = "/content/dataset_wm"
    IMG_TRAIN = os.path.join(DATASET_DIR, "images/train")
    IMG_VAL = os.path.join(DATASET_DIR, "images/val")
    LBL_TRAIN = os.path.join(DATASET_DIR, "labels/train")
    LBL_VAL = os.path.join(DATASET_DIR, "labels/val")

    for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
        os.makedirs(d, exist_ok=True)

    # 1. Master Template (Stempel Bulat Besar RFans & Teks Strip Diagonal & Sakerly)
    with open('core/b64_stamp.txt', 'r') as _f_s:
        B64_STAMP = _f_s.read().strip()
    with open('core/b64_sakerly.txt', 'r') as _f_sk:
        B64_SAKERLY = _f_sk.read().strip()

    stamp_raw = cv2.imdecode(np.frombuffer(base64.b64decode(B64_STAMP), np.uint8), cv2.IMREAD_UNCHANGED)
    stamp_img = stamp_raw[:, :, 3] if (len(stamp_raw.shape) == 3 and stamp_raw.shape[2] == 4) else stamp_raw

    sakerly_raw = cv2.imdecode(np.frombuffer(base64.b64decode(B64_SAKERLY), np.uint8), cv2.IMREAD_UNCHANGED)
    sakerly_img = sakerly_raw[:, :, 3] if (len(sakerly_raw.shape) == 3 and sakerly_raw.shape[2] == 4) else sakerly_raw

    # Buat template teks strip diagonal (@RFANS, TG@..., 电报直接搜索, t.me/SakerLy)
    text_strip_templates = []
    for txt in ["TG@RFans_Album_bot", "@RFANS CHANNEL", "电报直接搜索: @RFANS", "t.me/SakerLy", "t.me/album_cosplay"]:
        t_can = np.zeros((32, 280), dtype=np.uint8)
        cv2.putText(t_can, txt, (28, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, 255, 2, cv2.LINE_AA)
        cv2.circle(t_can, (14, 16), 10, 255, 2)
        text_strip_templates.append(('text_strip', t_can))

    # Kumpulkan semua template watermark (stempel + sakerly + teks strip + sampel user dari Google Drive)
    watermark_templates = [('stamp', stamp_img), ('sakerly', sakerly_img)] + text_strip_templates

    drive_samples = glob.glob(os.path.join(DRIVE_SAMPLES_DIR, "*.*"))
    local_samples = glob.glob(os.path.join(LOCAL_SAMPLES_DIR, "*.*"))
    all_user_samples = list(set(drive_samples + local_samples))

    print(f"📁 Memeriksa folder sampel Google Drive: {DRIVE_SAMPLES_DIR}")
    print(f"   Ditemukan {len(all_user_samples)} file crop watermark yang Anda kumpulkan.")

    for f in all_user_samples:
        u_raw = cv2.imread(f, cv2.IMREAD_UNCHANGED)
        if u_raw is None: continue
        # Ekstraksi otomatis bentuk huruf / watermark dari crop gambar:
        if len(u_raw.shape) == 3 and u_raw.shape[2] == 4:
            # Jika PNG transparan, gunakan langsung channel alphanya
            extracted_mask = u_raw[:, :, 3]
        else:
            u_gray = cv2.cvtColor(u_raw, cv2.COLOR_BGR2GRAY) if len(u_raw.shape) == 3 else u_raw
            # Ekstraksi guratan huruf kontras tinggi
            k_u = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
            top_u = cv2.morphologyEx(u_gray, cv2.MORPH_TOPHAT, k_u)
            _, stroke_u = cv2.threshold(top_u, 10, 255, cv2.THRESH_BINARY)
            if np.count_nonzero(stroke_u) < 40:
                _, stroke_u = cv2.threshold(u_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
            extracted_mask = stroke_u

        watermark_templates.append(('user_crop', extracted_mask))
        print(f"   [+] Sampel berhasil diekstrak huruf/bentuknya: {os.path.basename(f)}")

    # 2. Generator Sintetis 400 Gambar Latihan (Dengan Realistis Skin/Cloth & Sampel Negatif)
    print("[*] Membuat 400 data latihan cerdas (termasuk sampel negatif kulit/kain bersih agar tidak overfit)...")
    TOTAL_SAMPLES = 400
    for idx in range(TOTAL_SAMPLES):
        is_val = (idx % 5 == 0)
        img_out = IMG_VAL if is_val else IMG_TRAIN
        lbl_out = LBL_VAL if is_val else LBL_TRAIN

        H, W = 640, 640
        bg_type = random.choice(['skin_tone', 'cloth_dark', 'cloth_red', 'texture_wood', 'gradient'])
        canvas = np.zeros((H, W, 3), dtype=np.uint8)

        if bg_type == 'skin_tone':
            # Simulasi warna kulit (fair skin / warm tone)
            base_col = np.array([random.randint(170, 230), random.randint(185, 240), random.randint(210, 255)], dtype=np.uint8)
            canvas[:] = base_col
            noise = np.random.randint(-12, 12, (H, W, 3))
            canvas = np.clip(canvas.astype(int) + noise, 0, 255).astype(np.uint8)
        elif bg_type == 'cloth_red':
            # Simulasi kain kimono / baju merah
            canvas[:] = [random.randint(10, 40), random.randint(10, 40), random.randint(140, 230)]
        elif bg_type == 'cloth_dark':
            # Simulasi baju hitam / tatami gelap
            canvas[:] = [random.randint(20, 60), random.randint(20, 60), random.randint(20, 60)]
        elif bg_type == 'texture_wood':
            # Simulasi lantai kayu
            canvas[:] = [random.randint(30, 70), random.randint(60, 110), random.randint(100, 160)]
        else:
            c1 = np.random.randint(40, 220, 3)
            c2 = np.random.randint(40, 220, 3)
            X, Y = np.meshgrid(np.linspace(0, 1, W), np.linspace(0, 1, H))
            for c in range(3):
                canvas[:, :, c] = (c1[c] * (1 - Y) + c2[c] * Y).astype(np.uint8)

        labels = []
        # 25% Sampel Negatif (Tanpa Watermark sama sekali agar AI paham bahwa kulit/baju bersih BUKAN watermark!)
        is_negative_sample = (idx % 4 == 0)

        if not is_negative_sample:
            num_wm = random.choice([1, 2])
            for _ in range(num_wm):
                w_type, t_img = random.choice(watermark_templates)
                th, tw = t_img.shape[:2]

                target_w = random.randint(120, 260)
                target_h = int(th * (target_w / float(tw)))
                scaled = cv2.resize(t_img, (target_w, target_h))

                # Rotasi acak (-40 sampai +40 derajat)
                ang = random.uniform(-40.0, 40.0)
                M = cv2.getRotationMatrix2D((target_w / 2.0, target_h / 2.0), ang, 1.0)
                cos_a = np.abs(M[0, 0])
                sin_a = np.abs(M[0, 1])
                nw = int((target_h * sin_a) + (target_w * cos_a))
                nh = int((target_h * cos_a) + (target_w * sin_a))
                M[0, 2] += (nw / 2.0) - (target_w / 2.0)
                M[1, 2] += (nh / 2.0) - (target_h / 2.0)
                st_rot = cv2.warpAffine(scaled, M, (nw, nh))

                if nw >= W or nh >= H:
                    continue

                x = random.randint(5, W - nw - 5)
                y = random.randint(5, H - nh - 5)

                alpha = random.uniform(0.18, 0.70)
                roi = canvas[y:y+nh, x:x+nw].astype(float)
                mask_3c = np.repeat((st_rot > 60)[:, :, np.newaxis], 3, axis=2)
                canvas[y:y+nh, x:x+nw] = np.where(mask_3c, np.clip(roi * (1 - alpha) + 255 * alpha, 0, 255).astype(np.uint8), roi.astype(np.uint8))

                cx = (x + nw / 2.0) / float(W)
                cy = (y + nh / 2.0) / float(H)
                bw = nw / float(W)
                bh = nh / float(H)
                labels.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        # Simpan file gambar dan label YOLO
        img_fn = f"synth_{idx:04d}.jpg"
        cv2.imwrite(os.path.join(img_out, img_fn), canvas)
        with open(os.path.join(lbl_out, f"synth_{idx:04d}.txt"), "w") as f_lbl:
            f_lbl.write("\n".join(labels) + "\n")

    # 3. Buat File Konfigurasi YOLO dataset.yaml
    yaml_content = f"""path: {DATASET_DIR}
train: images/train
val: images/val
names:
  0: watermark
"""
    yaml_path = os.path.join(DATASET_DIR, "data.yaml")
    with open(yaml_path, "w") as f_y:
        f_y.write(yaml_content)

    # 4. Training YOLOv11 di GPU T4 (~1.5 Menit) - Transfer Learning dari Model Bawaan
    print("\n[*] Memulai Transfer Learning YOLOv11 di GPU NVIDIA T4 (~1.5 menit)...")
    from ultralytics import YOLO
    base_weights = "yolo11n.pt"
    model = YOLO(base_weights)
    model.train(
        data=yaml_path,
        epochs=12,
        imgsz=640,
        batch=16,
        device=0,
        verbose=False,
        save=True
    )

    # 5. Simpan Permanen ke Google Drive & Lokal
    trained_weights = "runs/detect/train/weights/best.pt"
    if os.path.exists(trained_weights):
        shutil.copy(trained_weights, LOCAL_MODEL)
        shutil.copy(LOCAL_MODEL, DRIVE_MODEL)
        print("\n" + "=" * 65)
        print("🎉 TRAINING SUKSES! MODEL TELAH TERSIMPAN PERMANEN!")
        print(f"💾 Lokasi Google Drive: {DRIVE_MODEL}")
        print("⚡ Mulai besok dan seterusnya, Anda TIDAK PERLU training ulang lagi!")
        print("=" * 65 + "\n")
        try:
            from google.colab import files
            files.download(LOCAL_MODEL)
        except Exception:
            pass
